# 03 — Black-Litterman

Notebooks 01 and 02 showed two ways for a mean-variance optimizer to fail:
feed it a noisy 252-day sample mean and it becomes an error-maximizer
(notebook 01); feed it a CAPM-implied mean and it algebraically collapses to
100% SPY (notebook 02, §5). Black-Litterman (BL) is the standard fix for
both: it doesn't estimate $\mu$ directly at all. It starts from a **prior**
mean implied by a reference portfolio, then **shrinks** that prior toward a
small number of explicit **views**, with the shrinkage weight set by how
confident the view is declared to be — never handing the optimizer a noisy
sample mean outright.

**The prior.** $\Pi = \delta\,\Sigma\, w_{EW}$, with $\delta = \text{sr\_ref}
/ \sigma_{EW}$ and $\text{sr\_ref}=0.3$ fixed *a priori* (a plausible
long-run diversified Sharpe, not fit to this sample). $w_{EW}$ — equal
weight — is chosen deliberately over a CAPM/SPY-implied prior: with no
views at all, BL's posterior mean collapses back to exactly $\Pi$ (shown in
§3), so an EW prior makes the **no-view portfolio the repo's own EW
benchmark** — every view is then measured against that benchmark directly. A
CAPM prior would instead make the no-view portfolio 100% SPY (§2), which
would just reproduce notebook 02's collapse and tell us nothing new about
the views themselves.

**The views.** One view per asset ($P=I$), $Q = \Pi + k\cdot
\text{sign}(\text{mom}_{12,1})\cdot\sigma$ — a **fixed, vol-scaled tilt** in
the direction of trailing 12-month-minus-1-month excess momentum, not the
trailing return itself (that would just be the sample-mean $\mu$ again, with
none of BL's shrinkage). View uncertainty follows He-Litterman:
$\Omega=\text{diag}(\text{diag}(\tau\Sigma))$, which makes $\tau$ **cancel
exactly** out of the posterior mean $\mu_{BL}$ (verified numerically in §3)
— so only $k$ and $\text{sr\_ref}$ (through $\delta$) actually matter for
the resulting weights. The optimizer is the same long-only tangency QP as
`MaxSharpe`, run on $\mu_{BL}$ against the **same** 252-day sample $\Sigma$
used by `GMV`/`MaxSharpe` (not a BL posterior covariance).

**On $k$.** The plan going in was to register $k=0.2$. A snapshot check at
2014-12-31 / 2017-12-31 / 2022-12-31 (all inside the train window) showed
that $k\ge0.2$ concentrates the portfolio hard at 2022-12-31 (effN drops
well below the 5-name floor used elsewhere in this notebook) — so $k=0.1$ is
registered instead, and $k=0.2$ is kept as a sensitivity check, not a second
"better" choice. §4 shows the mechanism directly.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import maplab as ml
import _nb03_helpers as h
from maplab import Panel, apply_style, BlackLitterman, GMV, MaxSharpe, EqualWeight, BetaTargetMinVar, backtest, summary, tangency_closed_form

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

print(f"prices: {prices.shape[0]:,} days × {prices.shape[1]} assets "
      f"({prices.index.min().date()} → {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

## Why not a CAPM prior

At the last rebalance on or before the train/test split (2022-12-31), take
`Sigma` from `BlackLitterman().posterior(...)` (the same production 252-day
$\Sigma$ used everywhere else) and build a **CAPM-implied** prior instead of
the EW-implied one: $\Pi_{CAPM} = \delta\,\Sigma\, e_{SPY}$ (all weight
notionally on the market). Its unconstrained tangency portfolio is exactly
100% SPY — this is the same algebraic identity as notebook 02 §5
($\Sigma^{-1}\Pi \propto \Sigma^{-1}\Sigma e_{SPY} = e_{SPY}$ exactly, for
*any* prior proportional to $\Sigma e_{SPY}$), not a new result. It's the
reason this notebook uses an EW-implied prior instead: a CAPM prior would
make every "no-view" comparison here just re-derive notebook 02's
collapse.

In [ ]:
rebal_le_split = ml.rebalance_dates(log_returns.index)
asof_split = rebal_le_split[rebal_le_split <= split_ts].max()
print(f"snapshot asof (last rebalance <= split): {asof_split.date()}")

bl_probe = BlackLitterman()
post_probe = bl_probe.posterior(panel, asof_split)
Sigma_probe = post_probe["Sigma"]
delta_probe = post_probe["delta"]

e_spy = pd.Series(0.0, index=ml.UNIVERSE)
e_spy["SPY"] = 1.0
Pi_capm = delta_probe * (Sigma_probe.to_numpy() @ e_spy.to_numpy())
Pi_capm = pd.Series(Pi_capm, index=ml.UNIVERSE)

w_capm_tangency = tangency_closed_form(Pi_capm, Sigma_probe, 0.0)
print(f"CAPM-implied prior -> unconstrained tangency SPY weight: {w_capm_tangency['SPY']:.6f}")
print(f"max |other assets|: {w_capm_tangency.drop('SPY').abs().max():.2e}")
print("Same collapse as notebook 02 §5 — the reason this notebook uses an EW-implied prior instead.")

## Mechanics at the split snapshot ($k=0.1$)

The full posterior at the same 2022-12-31 snapshot, registered $k=0.1$: the
EW-implied prior $\Pi$, asset vol $\sigma$, trailing 12-1 excess momentum and
its sign, the view vector $Q$, and the posterior mean $\mu_{BL}$ (all
annualized, %). Then three numerical checks the design promises:

1. **$k=0$ collapses to EW exactly** (long-only weights vs $w_{EW}$).
2. **$\tau$ cancels out of $\mu_{BL}$** ($\tau=0.01$ vs $\tau=1.0$).
3. **The closed-form identity holds**: $\Sigma^{-1}\mu_{BL} = \delta\,
   w_{EW} + k\,(\Sigma+D)^{-1}(\text{sign}\cdot\sigma)$, $D=\text{diag}(\text{diag}(\Sigma))$.

Finally, the **unconstrained** BL tangency (as in notebook 01's diagnostic
style) — this is *not* what `BlackLitterman.predict_weights` returns (that's
long-only), it's a diagnostic on how hard the views pull before the
long-only constraint clips them.

In [ ]:
bl01 = BlackLitterman(k=0.1)
post = bl01.posterior(panel, asof_split)

mech_table = h.mechanics_table(post)
print(f"delta = {post['delta']:.4f}   #Pi<0 = {int((post['pi'] < 0).sum())} "
      f"(expected: negative Pi_i = delta*cov(r_i, r_EW) is not a bug, e.g. UUP's "
      f"negative co-movement with the rest of the universe)")
mech_table.round(3)

In [ ]:
chk = h.design_checks(panel, asof_split, post)

# check 1: k=0 -> long-only weights == EW, mu_bl == pi
print(f"[k=0 -> EW] max|w_k0 - w_EW| = {chk['k0_dev']:.2e}")

# check 2: tau invariance
print(f"[tau invariance] max|mu_BL(tau=0.01) - mu_BL(tau=1.0)| = "
      f"{chk['tau_dev']:.2e}")

# check 3: closed-form identity
print(f"[closed-form identity] max|LHS - RHS| = {chk['identity_dev']:.2e}")

# unconstrained BL tangency
w_unc = chk["w_unc"]
top_long = w_unc.sort_values(ascending=False).head(3)
negatives = w_unc[w_unc < 0].sort_values()
top_short = negatives.head(3)
print(f"\n[unconstrained BL(k=0.1) tangency]  gross leverage = {w_unc.abs().sum():.3f}")
print(f"  largest longs:  {dict(top_long.round(3))}")
if len(top_short):
    print(f"  largest shorts: {dict(top_short.round(3))}")
else:
    print("  largest shorts: none")

## Cautionary snapshot — 2022-12-31 (no backtest at $k=0.4$)

2022 was a near-universal risk-off year across the 13-asset universe: 11/13
assets carry a **negative** 12-1 momentum sign (table in §3) — essentially
one macro bet ("risk-off"), not 13 independent signals. The portfolio is
**long-only and fully invested** with no cash asset to retreat to, so as $k$
grows the optimizer has nowhere to put a "risk-off" view except into
whichever few assets *are* positive-momentum (DBC, UUP) — concentration, not
diversification. This cell is a snapshot only; $k=0.4$ is never backtested
(§5 onward uses $k \in \{0.1, 0.2\}$).

In [ ]:
k_table, wtab = h.k_sensitivity(panel, asof_split)

print(k_table.round(3))
print()
print("long-only weights (%):")
print((100 * wtab).round(1))

Where do BL(0.1), BL(0.2), GMV, MaxSharpe and EW sit in (vol, μ) space at 2022-12-31? The y-axis is the 252d sample mean, NOT μ_BL: BL optimizes on its own posterior (next figure), so its position here is only illustrative.

In [ ]:
mu_2022, Sigma_2022 = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_split)
tickers_2022 = list(Sigma_2022.columns)
n_2022 = len(tickers_2022)

w_2022 = {
    "BL(0.1)": wtab["k=0.1"].reindex(tickers_2022),
    "BL(0.2)": wtab["k=0.2"].reindex(tickers_2022),
    "GMV": GMV()(panel, asof_split).reindex(tickers_2022),
    "MaxSharpe": MaxSharpe()(panel, asof_split).reindex(tickers_2022),
    "EqualWeight": pd.Series(1.0 / n_2022, index=tickers_2022),
}
bl_styles = {
    "BL(0.1)": {"marker": "s", "s": 110, "color": ml.FAMILY_COLORS[BlackLitterman.family], "label": "BL(0.1)"},
    "BL(0.2)": {"marker": "s", "s": 110, "color": "#c0392b", "label": "BL(0.2)"},
}

fig = ml.plotting.snapshot_map(
    Sigma_2022, mu_2022, {name: w.to_numpy() for name, w in w_2022.items()},
    title=f"Snapshot {asof_split.date()} -- BL(0.1) / BL(0.2) / GMV / MaxSharpe / EW",
    figsize=(10, 6), legend_outside=True, styles=bl_styles,
)
plt.show()

Prior vs posterior at 2022-12-31: the EW-implied prior Π (no-view BL = EW exactly) and the posterior μ_BL after the 12−1 momentum sign views. Negative-momentum assets are pulled down and positive-momentum assets (DBC, UUP at this date) pulled up; with no cash asset, the long-only optimizer can only express the risk-off tilt by concentrating into the latter.

In [ ]:
fig = h.prior_posterior_figure(panel, asof_split)
plt.show()

How far do the views move the weights from EW at 2022-12-31? (k=0.4 is snapshot only and never backtested.)

In [ ]:
fig = h.ew_vs_bl_weights(wtab)
plt.show()

Capital vs risk at 2022-12-31 (C5): bars = capital weight, faded bars = risk-contribution share, dashed = 1/13; a negative risk share means the asset hedges the rest of the portfolio.

In [ ]:
weights_c5 = {"BL(0.1)": w_2022["BL(0.1)"], "BL(0.2)": w_2022["BL(0.2)"], "GMV": w_2022["GMV"],
              "EW": w_2022["EqualWeight"]}
print(ml.diagnostics.diversification_table(weights_c5, Sigma_2022).to_string(float_format=lambda v: f"{v:.4f}"))

fig = ml.plotting.capital_vs_risk(weights_c5, Sigma_2022, title="Capital weight vs risk share, 2022-12-31 (sample Σ)")
plt.show()

## Backtest comparison

Six fresh strategy instances through `ml.backtest` exactly as in notebooks
01/02: `GMV`, `MaxSharpe`, `EqualWeight`, `BetaMinVar(0.3)`, `BL(k=0.1)`
(registered), `BL(k=0.2)` (sensitivity). Same 10bps one-way cost, monthly
drift-and-rebalance, common warm-up start.

In [ ]:
strategies = {
    "GMV": GMV(),
    "MaxSharpe": MaxSharpe(),
    "EqualWeight": EqualWeight(),
    "BetaMinVar(0.3)": BetaTargetMinVar(beta_target=0.3),
    "BL(0.1)": BlackLitterman(k=0.1),
    "BL(0.2)": BlackLitterman(k=0.2),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel]
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]
summary_table.round(4)

In [ ]:
bl01_strat = results["BL(0.1)"]["strat"]
bl02_strat = results["BL(0.2)"]["strat"]
for name, strat in [("BL(0.1)", bl01_strat), ("BL(0.2)", bl02_strat)]:
    print(f"{name}: fallback={len(strat.fallback_dates)}  retry={len(strat.retry_dates)}  "
          f"concentration={len(strat.concentration_dates)}")
    if strat.concentration_dates:
        print(f"  concentration dates: {[d.date() for d in strat.concentration_dates]}")

## BL(k=0.1) allocation through time

Target weights by asset group (as in notebook 02), plus effective N
($1/\sum w_i^2$) and max weight over time for `BL(0.1)` and `BL(0.2)`,
against the effN=5 concentration floor tracked in
`BlackLitterman.concentration_dates`.

Rows: effective N, max weight, UUP weight, and trailing 12-rebalance one-way turnover (fraction of the book per year, as percent=False; initial allocation excluded); EW dashed.

In [ ]:
fig = h.group_weights_figure(results, split_ts)
plt.show()

In [ ]:
fig = ml.plotting.concentration_panel(
    results, ["BL(0.1)", "BL(0.2)"], rows=("effN", "max_w", "highlight", "turnover"), ref="EqualWeight",
    split_ts=split_ts, figsize=(9.5, 14),
    suptitle="BL(0.1) vs BL(0.2): effN, max weight, UUP weight, turnover",
    titles=["Effective N", "Max single-asset weight", "UUP weight", "Trailing 12-rebalance turnover"], percent=False,
    effn_floor=5.0, xlabel=None,
    colors={"BL(0.1)": ml.FAMILY_COLORS[BlackLitterman.family], "BL(0.2)": "#c0392b"},
)
plt.show()

## Views vs EW: active return

Since the no-view BL posterior is exactly EW (§3, check 1), the cleanest way
to read what the momentum views are doing is `BL(k) - EW`'s **active
return**: annualized active return, tracking error, and information ratio
(active return / tracking error), full/train/test. Cumulative active return
plotted against the train/test split.

Exploratory (nb03 has no registration): these are the pairs the notebook's verdict rests on. Solid = β-adjusted (d_t − β̂·mkt_t), dashed = total.

In [ ]:
active_table, active_series = h.active_return_table(results, summary_table, split_ts)
active_table.round(4)

In [ ]:
c4_pairs = [("BL(0.1)", "EqualWeight"), ("BL(0.2)", "EqualWeight")]
c4_series = {(name, ref): active_series[name] for name, ref in c4_pairs}
c4_mkt = (simple_returns["SPY"] - rf_daily).dropna()
c4_diff_capm = ml.inference.diff_regression(c4_series, c4_pairs, c4_mkt)

fig = ml.plotting.cumulative_paired(
    c4_series, c4_pairs, split_ts,
    beta={f"{a} - {b}": c4_diff_capm.loc[f"{a} - {b}", "beta_diff"] for a, b in c4_pairs},
    mkt_excess=c4_mkt,
    title="Exploratory: cumulative BL − EW, solid = β-adjusted, dashed = total",
)
plt.show()

## Ex-post CAPM (plain OLS)

Same regression as notebook 02 §7: realized daily **simple** excess return
(`net - rf_daily`) on the market's simple excess return (`SPY - BIL`), for
`EqualWeight`, `BL(0.1)`, `BL(0.2)`. Same caveat: no HAC correction, so
$t$-stats are indicative, not exact.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_strategy_table = ml.inference.capm_table(results, ["EqualWeight", "BL(0.1)", "BL(0.2)"], MKT_simple, rf_daily)
capm_strategy_table.round(4)

How much of the active shortfall is lower beta?

In [ ]:
bl_pairs = [("BL(0.1)", "EqualWeight"), ("BL(0.2)", "EqualWeight")]
bl_paired, bl_paired_series = ml.inference.paired_table(results, bl_pairs, split_ts)
bl_diff_capm = ml.inference.diff_regression(bl_paired_series, bl_pairs, MKT_simple)
decomp = ml.inference.beta_decomposition(capm_strategy_table, bl_paired, bl_diff_capm, bl_pairs,
                                         results, rf_daily, MKT_simple)

for name, ref in bl_pairs:
    r = decomp[f"{name} - {ref}"]
    print(f"=== {name} vs EW (full window) ===")
    print(f"  d_alpha (ann)         = {r['d_alpha']:.4%}")
    print(f"  d_beta                = {r['d_beta']:.4f}")
    print(f"  ann mean MKT excess   = {r['mkt_ann_mean']:.4%}")
    print(f"  beta_part = d_beta * mkt_ann_mean = {r['beta_part']:.4%}")
    print(f"  d_alpha + beta_part   = {r['decomposed']:.4%}")
    print(f"  realized ann_active_return (§7, full) = {r['realized_diff_full']:.4%}")
    print(f"  identity check (0 by OLS construction) = {r['identity_check']:.4%}")
    print()

In [ ]:
print("Sharpe standard error ~ sqrt((1+SR^2/2)/T), T in years (same convention as notebooks 01/02):")
for name, r in results.items():
    net_full = r["net"]
    net_test = net_full.loc[net_full.index > split_ts]
    sr_full = float(summary_table.loc[(name, "full"), "sharpe"])
    sr_test = float(summary_table.loc[(name, "test (> split)"), "sharpe"])
    T_full = len(net_full) / ml.TRADING_DAYS
    T_test = len(net_test) / ml.TRADING_DAYS
    se_full = ml.inference.sharpe_se(sr_full, T_full)
    se_test = ml.inference.sharpe_se(sr_test, T_test)
    print(f"  {name:16s} full: SR={sr_full:.2f} SE={se_full:.2f}   |   test: SR={sr_test:.2f} SE={se_test:.2f}")

**TAKEAWAY**

- **Design checks hold to floating-point precision.** A CAPM-implied prior gives an
  unconstrained tangency of exactly 100% SPY (others ≤ 2.04e-15) — notebook 02's
  collapse, so a CAPM prior would tell us nothing about the views here. k=0 reproduces EW
  (max|Δw| = 3.66e-07, the SLSQP tolerance floor), τ cancels (max diff 6.94e-18), and the
  closed-form identity holds (max error 2.97e-15).
- **Negative Π is expected.** Only UUP (Π = −1.81%) is negative at 2022-12-31: Π_i ∝
  cov(r_i, r_EW), and UUP co-moves negatively with the EW book.
- **k is a concentration knob.** At 2022-12-31, long-only effN goes 7.58 → 3.47 → 1.78 and
  unconstrained gross leverage 1.131 → 2.038 → 68.899 for k = 0.1 / 0.2 / 0.4. k was shrunk
  from the planned 0.2 to 0.1 on this train-window snapshot; 0.4 is never backtested.
- **No Sharpe difference at the ~2 SE bar.** Full window (SE ≈ 0.26–0.28): GMV 0.62,
  MaxSharpe 0.72, EW 0.74, BetaMinVar(0.3) 0.86, BL(0.1) 0.64, BL(0.2) 0.58; the widest gap
  (BetaMinVar(0.3) vs BL(0.2), 0.28) is about 1 SE. Test window (SE ≈ 0.58–0.70): widest
  gap 0.63 (MaxSharpe/BetaMinVar(0.3) at 1.13 vs BL(0.2) at 0.50), also about 1 SE.
  (Single-strategy SEs; the SE of a difference between correlated strategies is not
  computed here.)
- **The views lag EW in every window, but not significantly, and not because of costs.**
  BL(0.1) − EW: −1.36%/yr full (t = −1.78), −1.62%/yr test (t = −1.11); BL(0.2) − EW:
  −1.97%/yr full (t = −1.66), −3.01%/yr test (t = −1.33). The cost drag difference is
  only 0.23%/yr (BL(0.1), full), so gross of costs the gap is still −1.14%/yr.
- **Most of the shortfall is lower beta, not negative alpha.** Full window, BL(0.1) vs EW:
  Δβ = −0.0655, which at the realized 14.72%/yr market excess accounts for −0.96%/yr;
  Δα = −0.40%/yr. No alpha is significant (EW t = 0.49, BL(0.1) t = 0.14, BL(0.2) t = 0.32;
  plain OLS, no HAC).
- **Concentration is episodic, not only 2022.** effN < 5 on 16/208 rebalances for BL(0.1)
  and 66/208 for BL(0.2), with no fallbacks or retries. BL(0.1)'s dates cluster in
  Dec 2013 (1), Jan–Apr 2014 (4), Dec 2015 (1), Jan 2016 (1), Jul/Oct/Nov/Dec 2018 (4),
  May 2022 (1), and Apr/May/Jul/Nov 2023 (4). The 2022-12-31 snapshot shows one mechanism —
  absolute risk-off views with no cash asset force concentration into the few
  positive-momentum assets (DBC, UUP) — which motivates the Phase 3 overlay; whether the
  other episodes share that cause is not tested here.
